## 450 QA pairs into 250 QA pairs (5 category equally)

In [ ]:
import json
from pathlib import Path
from collections import Counter, defaultdict
import random

IN_PATH = Path("./test_advanced_450_2.json")
OUT_PATH = Path("./test_advanced_250_2_processed.json")

TYPES = ["事实提取", "列举枚举", "比较计算", "判断验证", "推理分析"]
PER_TYPE = 50
SEED = 42

data = json.loads(IN_PATH.read_text(encoding="utf-8"))
print(f"Loaded {len(data)} questions from {IN_PATH}")

#de-duplicate by question (keep first)
seen_q = set()
dedup = []
for x in data:
    q = (x.get("question","").strip())
    if not q or q in seen_q:
        continue
    seen_q.add(q)
    dedup.append(x)

by_type = defaultdict(list)
for x in dedup:
    t = (x.get("type","").strip())
    by_type[t].append(x)

# sanity check
missing = [t for t in TYPES if len(by_type[t]) < PER_TYPE]
if missing:
    raise RuntimeError("Not enough samples for types: " + ", ".join(
        f"{t} ({len(by_type[t])}<{PER_TYPE})" for t in missing
    ))

random.seed(SEED)
out = []
for t in TYPES:
    out.extend(random.sample(by_type[t], PER_TYPE))

random.shuffle(out)

OUT_PATH.write_text(json.dumps(out, ensure_ascii=False, indent=2), encoding="utf-8")

print("Wrote:", OUT_PATH)
print("Total:", len(out))
print("Type counts:", dict(Counter((x.get('type') or '').strip() for x in out)))

## Format the questions

In [2]:
import json, re
from pathlib import Path

IN_PATH = Path("./test_advanced_250_2_processed.json")
OUT_PATH = Path("./test_advanced_250_2_processed_fmt.json")

# 这些前缀的冒号是模板语义的一部分，不删
KEEP_COLON_PREFIXES = (
    "归因分析：", "原因解释：", "差异分析：", "风险评估：", "效率分析：",
    "杠杆评估：", "库存分析：", "ROE拆解：", "战略评估：", "质量分析：",
)

def normalize_colon(s: str) -> str:
    # 统一中英文冒号，便于匹配
    return s.replace("：", ":")

def restore_colon_style(s: str) -> str:
    # 如果你希望最终保持中文冒号风格，可把前缀的 ":" 换回 "："
    # 这里简单把所有 ":" 换回 "："（也可以只换前缀）
    return s.replace(":", "：")

def fmt_question(q: str) -> str:
    if not q:
        return q
    raw = q
    q = normalize_colon(q).strip()

    # ---- Rule 1: "公司名:公司名..." 去重（只处理开头位置）----
    # 例：兆易创新:兆易创新2024年年度报告... -> 兆易创新2024年年度报告...
    q = re.sub(r"^(.{2,30}):\1", r"\1", q)

    # ---- Rule 2: 去掉公司名与年份/年度报告之间的多余冒号 ----
    # 只在“不是分析前缀句式”时应用，避免误删“归因分析：...”
    if not any(raw.startswith(pfx) for pfx in KEEP_COLON_PREFIXES):
        # 常见：...公司名:2024年年度报告...
        q = re.sub(r"([^\s:]{2,30}):(?=(20\d{2}|报告期内|年度))", r"\1", q, count=1)

        # 常见：请列举公司名:2024年年度报...
        q = re.sub(r"(请列举[^\s:]{2,30}):(?=(20\d{2}|报告期内|年度))", r"\1", q, count=1)

    # 可选：如果出现连续多个冒号（通常是噪声），压缩为一个
    q = re.sub(r":{2,}", ":", q)

    # 还原中文冒号（如果你想保留中文标点风格）
    q = restore_colon_style(q)

    return q.strip()

data = json.loads(IN_PATH.read_text(encoding="utf-8"))
for x in data:
    x["question"] = fmt_question(x.get("question", ""))

OUT_PATH.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")
print("Wrote:", OUT_PATH)

Wrote: test_advanced_250_2_processed_fmt.json


In [4]:
import json
import re
from pathlib import Path

IN_PATH = Path("./test_advanced_250_2.json")
OUT_PATH = Path("./test_advanced_250_2.formatted.json")

def format_question(q: str) -> str:
    if not q:
        return q

    q = q.strip()

    # Rule 1: 2024年年度报告 -> 2024年度
    q = re.sub(r"(\d{4})年年度报告", r"\1年度", q)

    # Rule 2: 2024年度2024年度 -> 2024年度
    q = re.sub(r"(\d{4})年度\1年度", r"\1年度", q)

    # 可选：处理极端重复（比如连续3次）
    q = re.sub(r"(\d{4})年度(?:\1年度)+", r"\1年度", q)

    # Rule 3: 结尾标点统一
    if q.endswith("。"):
        q = q[:-1] + "？"
    elif q.endswith("."):
        q = q[:-1] + "?"
    return q

data = json.loads(IN_PATH.read_text(encoding="utf-8"))
for x in data:
    x["question"] = format_question(x.get("question", ""))

OUT_PATH.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")
print("Wrote:", OUT_PATH)

Wrote: test_advanced_250_2.formatted.json


In [5]:
import json
import re
from pathlib import Path

IN_PATH = Path("./test_advanced_250_2.formatted.json")
OUT_PATH = Path("./test_advanced_250_2.formatted.company_clean.json")

re_dup_company = re.compile(r"([\u4e00-\u9fffA-Za-z0-9]{2,30})\s*[:：]\s*\1")
re_company_year_colon = re.compile(r"([\u4e00-\u9fffA-Za-z0-9]{2,30})\s*[:：]\s*(\d{4}年度)")

def clean_question(q: str) -> str:
    if not q:
        return q
    q = q.strip()

    # A) 公司：公司 -> 公司
    q = re_dup_company.sub(r"\1", q)

    # B) 公司：2024年度 -> 公司2024年度（保留前缀如“归因分析：”的冒号）
    q = re_company_year_colon.sub(r"\1\2", q)

    # 可选：清理极端“：：”
    q = re.sub(r"[:：]{2,}", "：", q)
    return q

data = json.loads(IN_PATH.read_text(encoding="utf-8"))
changed = 0
for x in data:
    old = x.get("question", "")
    new = clean_question(old)
    if new != old:
        changed += 1
    x["question"] = new

OUT_PATH.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")
print("Wrote:", OUT_PATH)
print("Changed questions:", changed, "/", len(data))

Wrote: test_advanced_250_2.formatted.company_clean.json
Changed questions: 64 / 250


# Statistics Validation

In [6]:
import json
from pathlib import Path
from collections import Counter
import re

IN_PATH = Path("./test_advanced_250_2.json")

data = json.loads(IN_PATH.read_text(encoding="utf-8"))
print("Total:", len(data))

# type counts
cnt = Counter((x.get("type") or "").strip() for x in data)
print("Type counts:")
for k, v in cnt.most_common():
    print(f"  {k!r}: {v}")

# validation checks
required = ["filename", "page", "question", "answer", "type"]

missing = []
bad_page = []
bad_fn = []
bad_q_end = []
questions = []

def extract_company_from_filename(fn: str) -> str:
    fn = (fn or "").strip()
    if not fn:
        return ""
    # take left side of Chinese colon if present: "公司：xxxx.pdf"
    if "：" in fn:
        company = fn.split("：", 1)[0].strip()
        return company
    # fallback: remove extension and trailing year/report words
    stem = fn
    stem = re.sub(r"\.pdf$", "", stem, flags=re.I).strip()
    return stem 

companies = []
for x in data:
    companies.append(extract_company_from_filename(x.get("filename", "")))
    
comp_cnt = Counter([c for c in companies if c])
print("Unique companies:", len(comp_cnt))
print("Top 10 companies by question count:")
for c, n in comp_cnt.most_common(10):
    print(f"  {c}: {n}")

for i, x in enumerate(data):
    miss = [k for k in required if k not in x]
    if miss:
        missing.append((i, miss))

    if not isinstance(x.get("page"), int):
        bad_page.append((i, x.get("page")))

    if not (x.get("filename") or "").strip():
        bad_fn.append(i)

    q = (x.get("question") or "").strip()
    questions.append(q)
    if q.endswith("。") or q.endswith("."):
        bad_q_end.append((i, q[-1]))

qcnt = Counter(q for q in questions if q)
dup = [(q, c) for q, c in qcnt.items() if c > 1]

print("Missing-field rows:", len(missing))
print("Non-int page rows:", len(bad_page))
print("Empty filename rows:", len(bad_fn))
print("Questions ending with 。 or .:", len(bad_q_end))
print("Unique questions:", len(qcnt))
print("Duplicate questions:", len(dup))
if dup[:5]:
    print("Dup examples:", dup[:5])


Total: 250
Type counts:
  '推理分析': 50
  '列举枚举': 50
  '判断验证': 50
  '事实提取': 50
  '比较计算': 50
Unique companies: 40
Top 10 companies by question count:
  浙文互联: 14
  新易盛: 9
  中微公司: 9
  阿里巴巴-SW: 9
  海天味业: 8
  隆基绿能: 8
  鼎龙股份: 8
  快手-W: 8
  中科曙光: 8
  兆易创新: 7
Missing-field rows: 0
Non-int page rows: 0
Empty filename rows: 0
Questions ending with 。 or .: 0
Unique questions: 250
Duplicate questions: 0


In [11]:
# Merge 2 json files 
import json
from pathlib import Path

IN_PATH1 = Path("./test_advanced_250_2.json")
IN_PATH2 = Path("./gold_standard.json")

data1 = json.loads(IN_PATH1.read_text(encoding="utf-8"))
data2 = json.loads(IN_PATH2.read_text(encoding="utf-8"))

# Merge the two lists
merged_data = data1 + data2

# Save the merged data
OUT_PATH = Path("./gold_standard_500.json")
OUT_PATH.write_text(json.dumps(merged_data, ensure_ascii=False, indent=2), encoding="utf-8")

100666

In [3]:
import json
from pathlib import Path
from collections import Counter

IN_PATH = Path("./gold_standard_500.json")

data = json.loads(IN_PATH.read_text(encoding="utf-8"))
print("Total:", len(data))

# type counts
cnt = Counter((x.get("type") or "").strip() for x in data)
print("Type counts:")
for k, v in cnt.most_common():
    print(f"  {k!r}: {v}")

# validation checks
required = ["filename", "page", "question", "answer", "type"]

missing = []
bad_page = []
bad_fn = []
bad_q_end = []
questions = []

for i, x in enumerate(data):
    miss = [k for k in required if k not in x]
    if miss:
        missing.append((i, miss))

    if not isinstance(x.get("page"), int):
        bad_page.append((i, x.get("page")))

    if not (x.get("filename") or "").strip():
        bad_fn.append(i)

    q = (x.get("question") or "").strip()
    questions.append(q)
    if q.endswith("。") or q.endswith("."):
        bad_q_end.append((i, q[-1]))

qcnt = Counter(q for q in questions if q)
dup = [(q, c) for q, c in qcnt.items() if c > 1]

print("Missing-field rows:", len(missing))
print("Non-int page rows:", len(bad_page))
print("Empty filename rows:", len(bad_fn))
print("Questions ending with 。 or .:", len(bad_q_end))
print("Unique questions:", len(qcnt))
print("Duplicate questions:", len(dup))
if dup[:5]:
    print("Dup examples:", dup[:5])

Total: 500
Type counts:
  '推理分析': 100
  '列举枚举': 100
  '判断验证': 100
  '事实提取': 100
  '比较计算': 100
Missing-field rows: 0
Non-int page rows: 0
Empty filename rows: 0
Questions ending with 。 or .: 0
Unique questions: 500
Duplicate questions: 0


In [15]:
import json
from pathlib import Path

p = Path("./gold_standard_500.json")
out = Path("./gold_standard_500.qmark.json")

data = json.loads(p.read_text(encoding="utf-8"))

n = 0
for x in data:
    q = (x.get("question") or "").strip()
    if q.endswith("。"):
        x["question"] = q[:-1] + "？"
        n += 1
    elif q.endswith("."):
        x["question"] = q[:-1] + "?"
        n += 1

out.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")
print("Updated:", n)
print("Wrote:", out)

Updated: 123
Wrote: gold_standard_500.qmark.json
